In [1]:
!pip -q install -U "datasets>=3.0.0" "transformers>=4.46.0" "peft>=0.13.0" "accelerate>=1.0.0" "torchao>=0.16.0" safetensors tqdm

from google.colab import drive
drive.mount("/content/drive")

import os, json, glob, shutil, random, hashlib
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

os.environ["TOKENIZERS_PARALLELISM"] = "false"

root = "/content/drive/MyDrive/Test"
out_dir = os.path.join(root, "Python", "codeparrot_python_aux")
ckpt_dir = os.path.join(out_dir, "checkpoints")
best_dir = os.path.join(out_dir, "best")
last_dir = os.path.join(out_dir, "last")
os.makedirs(out_dir, exist_ok=True)

model_name = "EleutherAI/pythia-1.4b-deduped"
dataset_name = "codeparrot/codeparrot-clean"

seed = 42
batch_size = 16
lr = 5e-5
max_length = 256
epochs = 5
weight_decay = 0.01
warmup_ratio = 0.05
grad_accum = 1
max_grad_norm = 1.0

lora_r = 64
lora_alpha = 64
lora_dropout = 0.10
lora_modules = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

n_train = 20_000
n_val = 1_000
max_chars = 4_000

random.seed(seed)
torch.manual_seed(seed)

def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def dump(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

raw = load_dataset(dataset_name, split="train", streaming=True)
raw = raw.shuffle(buffer_size=10_000, seed=seed)

train_rows, val_rows, seen = [], [], set()

for ex in raw:
    text = ex["content"].strip()
    if not text:
        continue

    text = text[:max_chars]
    h = sha(text)

    if h in seen:
        continue
    seen.add(h)

    row = {"text": text, "sha256": h}
    bucket = int(h[:8], 16) % 100

    if bucket < 5:
        if len(val_rows) < n_val:
            val_rows.append(row)
    else:
        if len(train_rows) < n_train:
            train_rows.append(row)

    if len(train_rows) >= n_train and len(val_rows) >= n_val:
        break

val_hashes = {r["sha256"] for r in val_rows}
train_rows = [r for r in train_rows if r["sha256"] not in val_hashes]

train_hashes = {r["sha256"] for r in train_rows}
overlap = train_hashes & val_hashes

print("train:", len(train_rows))
print("val:", len(val_rows))
print("train/val overlap:", len(overlap))

dump({
    "split_protection": {
        "unit": "sha256 text hash",
        "priority": "val wins > train",
        "train_rows": len(train_rows),
        "val_rows": len(val_rows),
        "train_val_overlap": len(overlap),
    },
    "train_sha256": sorted(train_hashes),
    "val_sha256": sorted(val_hashes),
}, os.path.join(out_dir, "codeparrot_split_hashes.json"))

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

tok = AutoTokenizer.from_pretrained(model_name)
tok.pad_token = tok.eos_token

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=max_length)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text", "sha256"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["text", "sha256"])
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

base = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
base.config.pad_token_id = tok.eos_token_id
base.gradient_checkpointing_enable()
base.config.use_cache = False

model = get_peft_model(
    base,
    LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=lora_modules,
    ),
)
model.enable_input_require_grads()
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=ckpt_dir,
    seed=seed,
    data_seed=seed,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=lr,
    weight_decay=weight_decay,
    warmup_ratio=warmup_ratio,
    max_grad_norm=max_grad_norm,
    fp16=True,
    bf16=False,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

trainer.train()

metrics = trainer.evaluate()
metrics["perplexity"] = float(torch.exp(torch.tensor(metrics["eval_loss"])))
dump(metrics, os.path.join(out_dir, "eval_metrics.json"))

print("eval_loss:", metrics["eval_loss"])
print("perplexity:", metrics["perplexity"])

shutil.rmtree(best_dir, ignore_errors=True)
trainer.model.save_pretrained(best_dir, safe_serialization=True)
tok.save_pretrained(best_dir)

ckpts = sorted(
    glob.glob(os.path.join(ckpt_dir, "checkpoint-*")),
    key=lambda p: int(p.rsplit("-", 1)[1]),
)
last_ckpt = ckpts[-1]

shutil.rmtree(last_dir, ignore_errors=True)
os.makedirs(last_dir, exist_ok=True)

for name in ["adapter_model.safetensors", "adapter_config.json", "README.md"]:
    src = os.path.join(last_ckpt, name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(last_dir, name))

tok.save_pretrained(last_dir)

summary = {
    "model_name": model_name,
    "dataset_name": dataset_name,
    "data_dir": None,
    "seed": seed,
    "n_train_files": n_train,
    "n_val_files": n_val,
    "num_train_rows": len(train_rows),
    "num_val_rows": len(val_rows),
    "max_chars_per_file": max_chars,
    "max_length": max_length,
    "epochs": epochs,
    "batch_size": batch_size,
    "lr": lr,
    "weight_decay": weight_decay,
    "warmup_ratio": warmup_ratio,
    "grad_accum": grad_accum,
    "max_grad_norm": max_grad_norm,
    "fp16": True,
    "bf16": False,
    "lora_r": lora_r,
    "lora_alpha": lora_alpha,
    "lora_dropout": lora_dropout,
    "lora_target_modules": lora_modules,
    "split_protection": {
        "unit": "sha256 text hash",
        "train_val_overlap": len(overlap),
    },
    "best_eval_loss": trainer.state.best_metric,
    "eval_metrics": metrics,
    "adapter_best_dir": best_dir,
    "adapter_last_dir": last_dir,
    "last_checkpoint_source": last_ckpt,
    "selection_rule": "best adapter is selected by lowest eval_loss",
}

dump(summary, os.path.join(out_dir, "train_summary.json"))
dump(summary, os.path.join(out_dir, "train_config.json"))

print("saved best LoRA delta to:", best_dir)
print("saved last LoRA delta to:", last_dir)
print("summary:", os.path.join(out_dir, "train_summary.json"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 47.2 MB/s eta 0:00:00
Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

train: 20000
val: 1000
train/val overlap: 0


...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 50,331,648 || all params: 1,464,979,456 || trainable%: 3.4357


Epoch,Training Loss,Validation Loss
1,1.320327,1.271520
2,1.259488,1.258123
3,1.197381,1.255785
4,1.146232,1.260299
5,1.112997,1.265916


Training Loss,Validation Loss,Epoch
1.112997,1.255785,5


eval_loss: 1.2557847499847412
perplexity: 3.510592222213745
saved best LoRA delta to: /content/drive/MyDrive/Test/Python/codeparrot_python_aux/best
saved last LoRA delta to: /content/drive/MyDrive/Test/Python/codeparrot_python_aux/last
summary: /content/drive/MyDrive/Test/Python/codeparrot_python_aux/train_summary.json
